# KrVoiceAI · Colab GPU Worker

这台 Colab 当 **GPU worker**:连 `krvoice.cicy-ai.com` 的任务队列,拉任务 → GPU 出片 → 回传成片。

**用户在口播工坊(krvoice.cicy-ai.com)提交任务,这里自动领走出片。** 素材从 OSS 拉,不需要 Google Drive。

**先选 GPU**:`代码执行程序 → 更改运行时类型 → A100 或 L4`(LatentSync 512 需 24G+,L4/A100 都行,T4 不够)。然后从上到下逐格运行。

In [ ]:
# 1. 确认 GPU(A100 或 L4,显存 ≥23G)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. 拉最新代码
%cd /content
import os
if os.path.isdir('KrVoiceAI'):
    %cd KrVoiceAI
    !git pull -q
else:
    !git clone -q https://github.com/cicy-ai/KrVoiceAI.git
    %cd KrVoiceAI
!git log --oneline -1

In [ ]:
# 2.5 先装好环境(clone LatentSync + 依赖 + 模型 ~5GB,首次约 15-25 分钟)
#     装在固定位置 /content/LatentSync,装一次、之后所有出片复用,不重装。
#     日志实时滚在本格;看到「✅ 环境就绪」+ CUDA 可用 True 就装好了,再去第3格起 worker。
!cd /content/KrVoiceAI && SETUP_ONLY=1 bash colab/latentsync.sh

In [ ]:
# 3. 起 worker(前台跑!日志实时滚到本格,每一行你都看得到)
#    这一格会「一直运行」= worker 在轮询队列;想停就点本格的停止方块。
#    worker 自动认 GPU:有 GPU→真出片(LatentSync 数字人);没 GPU→假出片(造占位片,只验证链路闭环)。
#    命令永远这一条,不用改。切了 GPU 就是真出片,切了 CPU 就是验链路。
!cd /content/KrVoiceAI && pip install -q requests edge-tts 2>&1 | tail -1
!cd /content/KrVoiceAI && QUEUE_URL=https://krvoice.cicy-ai.com python -u colab/pull_worker.py

In [ ]:
# 4. 看 worker 状态 / 出片进度(反复运行本格)
import os
print(open('/content/pull_worker.log').read()[-2500:] if os.path.exists('/content/pull_worker.log') else '(worker 未启动,先跑第3格)')

---
- worker 起来后会每几秒轮询队列;用户在口播工坊点「生成」→ 这里自动领任务出片。
- 改代码后:重跑**第2格**(git pull)+ **第3格**(重启 worker)。
- 页面刷新/断连不影响已在跑的 worker(setsid 脱离);运行时被回收才会停。